# Manual Track Curation — windowed ROI variant

For the three implantation-stage windows on dataset003 (250914_stack5), each
a small (t, z) region rather than the full 121-timepoint stack. Loads and
curates just one window at a time — set `WINDOW_NAME` below.

Unlike `04_curate_tracks.ipynb`, this expresses z in **absolute (original,
pre-crop) coordinates** — the same real depth at every timepoint, since the
microscope images the same physical z-range every frame and the per-frame
crop is just a storage optimisation, not a spatial alignment. Each frame's
own crop offset (`first_z` + any `debris_shift_slices`, from `crop_info.json`)
is added back before stacking, so a single z-range means the same thing for
every timepoint in the window — no risk of flattening real z-motion the way
aligning frames to their own detected-signal start would.

Same curation GUI/workflow as `04_curate_tracks.ipynb` — see that notebook's
intro for the correction format.

In [73]:
import sys
from pathlib import Path
import yaml

REPO_ROOT   = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path('/mnt/md0/elysse/code/embryo_image_analysis')
CONFIG_PATH = REPO_ROOT / 'configs' / 'tracking' / 'dataset003_icm_te_250914_stack5.yaml'

# Windows are (t_start, t_end, z_lo_um, z_hi_um) in ABSOLUTE (original-stack)
# coordinates - not each frame's own local crop-relative z. z-bounds below
# were read off a single representative frame's *local* display before the
# absolute-z alignment existed, so treat them as a starting point: after
# loading, scroll the aligned stack and nudge if the real signal band looks
# offset from what's expected.
WINDOWS = {
    'A': {'t_start': 40, 't_end': 50, 'z_lo_um': 148, 'z_hi_um': 172},  # confirmed via napari local-z 98-112 (+60 shared_origin)
    'B': {'t_start': 70, 't_end': 80, 'z_lo_um': 162,  'z_hi_um': 180},
    'C': {'t_start': 90, 't_end': 95, 'z_lo_um': 57,  'z_hi_um': 67},
}
WINDOW_NAME = 'B'  # edit to switch window

TRACK_SOURCE = 'btrack'  # 'linked' or 'btrack' - matches 04_curate_tracks.ipynb
LINKED_VERSION_OVERRIDE = None

# To continue editing a version you already saved from THIS notebook (e.g.
# after saving 'tracks_linked_c1_winA.csv' and wanting to make a c2 pass on
# top of it), set this to that version string instead of starting fresh from
# TRACK_SOURCE's raw file. Bump OUTPUT_VERSION_OVERRIDE too so the next save
# doesn't overwrite what you just loaded.
RELOAD_VERSION = 'linked_c3_winB'          # e.g. 'linked_c1_winA'
OUTPUT_VERSION_OVERRIDE = 'linked_c4_winB' # e.g. 'linked_c2_winA' - defaults to config's output_version + _win{WINDOW_NAME} if None

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

RAW_DIR    = Path(cfg['paths']['raw_dir'])
LABEL_DIR  = Path(cfg['paths']['label_dir'])
RAW_GLOB   = cfg['paths']['raw_glob']
LABEL_GLOB = cfg['paths']['label_glob']
VX_Z, VX_Y, VX_X = cfg['microscopy']['voxel_size_zyx']
OUT_VERSION = OUTPUT_VERSION_OVERRIDE or (cfg['tracking']['output_version'] + f'_win{WINDOW_NAME}')

window = WINDOWS[WINDOW_NAME]
T_START, T_END = window['t_start'], window['t_end']
Z_LO_UM, Z_HI_UM = window['z_lo_um'], window['z_hi_um']

if TRACK_SOURCE == 'linked':
    TRACKS_DIR = Path(cfg['paths']['linked_tracks_dir'])
    VERSION    = LINKED_VERSION_OVERRIDE or cfg['tracking']['input_version']
    TRACKS_CSV = TRACKS_DIR / f'tracks_{VERSION}.csv'
elif TRACK_SOURCE == 'btrack':
    TRACKS_DIR = Path(cfg['paths']['btrack_output_dir'])
    VERSION    = 'v1'
    TRACKS_CSV = TRACKS_DIR / f'tracks_{VERSION}_raw.csv'
else:
    raise ValueError(f"Unknown TRACK_SOURCE: {TRACK_SOURCE!r}")

if RELOAD_VERSION:
    # Overrides the source above entirely - a saved windowed-notebook output
    # already has track_id/t/label_id/z_um/y_um/x_um in the same shape this
    # notebook expects, so it can be re-loaded and re-processed (re-aligned,
    # re-filtered to the window) exactly like a fresh source.
    TRACKS_CSV = TRACKS_DIR / f'tracks_{RELOAD_VERSION}.csv'

print(f'Config:      {CONFIG_PATH}')
print(f'Window:      {WINDOW_NAME} (t{T_START}-{T_END}, z {Z_LO_UM}-{Z_HI_UM} um absolute)')
print(f'Track source: {TRACK_SOURCE}')
print(f'Input CSV:   {TRACKS_CSV}')
print(f'Output CSV:  {TRACKS_DIR / f"tracks_{OUT_VERSION}.csv"}')

Config:      /mnt/md0/elysse/code/embryo_image_analysis/configs/tracking/dataset003_icm_te_250914_stack5.yaml
Window:      B (t70-80, z 162-180 um absolute)
Track source: btrack
Input CSV:   /mnt/md0/elysse/nnUNet/inference/Dataset003_icm_te/250914_stack5/results/tracks/btrack/tracks_linked_c3_winB.csv
Output CSV:  /mnt/md0/elysse/nnUNet/inference/Dataset003_icm_te/250914_stack5/results/tracks/btrack/tracks_linked_c4_winB.csv


In [74]:
import glob
import json
import re
import numpy as np
import pandas as pd
import napari
from skimage.io import imread, imsave

crop_info_path = RAW_DIR.parent.parent / 'crop_info.json'
with open(crop_info_path) as f:
    crop_info = json.load(f)

def absolute_first_z(tp_id):
    """Original-stack z-index of local index 0 in the currently-saved crop
    for this timepoint, after any debris correction."""
    e = crop_info[tp_id]
    return e['first_z'] + e.get('debris_shift_slices', 0)

# Load tracks, restrict to the window's t-range up front (cheap, avoids
# dragging the full-dataset track table through everything below).
tracks_df = pd.read_csv(TRACKS_CSV)
tracks_df = tracks_df[(tracks_df['t'] >= T_START) & (tracks_df['t'] <= T_END)].copy()
print(f'Loaded {tracks_df["track_id"].nunique()} tracks, {len(tracks_df)} rows in t{T_START}-{T_END} '
      f'from {TRACKS_CSV.name} (source={TRACK_SOURCE})')

has_orig = 'z_um_orig' in tracks_df.columns
if not has_orig and 'label_id' in tracks_df.columns:
    # Same btrack-coordinate recovery as 04_curate_tracks.ipynb - see that
    # notebook's version of this cell for the full explanation.
    raw_df = pd.read_csv(cfg['paths']['features_csv'])
    raw_df = raw_df[(raw_df['t'] >= T_START) & (raw_df['t'] <= T_END)][['t', 'label_id', 'z_um', 'y_um', 'x_um']]
    tracks_df = tracks_df.rename(columns={'z_um': 'z_um_reg', 'y_um': 'y_um_reg', 'x_um': 'x_um_reg'})
    tracks_df['label_id'] = tracks_df['label_id'].astype('Int64')
    tracks_df = tracks_df.merge(
        raw_df.rename(columns={'z_um': 'z_um_orig', 'y_um': 'y_um_orig', 'x_um': 'x_um_orig'}),
        on=['t', 'label_id'], how='left',
    )
    for orig_col, reg_col in [('z_um_orig', 'z_um_reg'), ('y_um_orig', 'y_um_reg'), ('x_um_orig', 'x_um_reg')]:
        tracks_df[orig_col] = tracks_df[orig_col].fillna(tracks_df[reg_col])
    has_orig = tracks_df['z_um_orig'].notna().any()

z_col, y_col, x_col = ('z_um_orig', 'y_um_orig', 'x_um_orig') if has_orig else ('z_um', 'y_um', 'x_um')

# tracks_df's z is local-crop-relative per timepoint (already corrected for
# debris, per scripts/fix_debris_crop_offsets.py) - convert to absolute
# original-stack z so it's directly comparable to Z_LO_UM/Z_HI_UM and to the
# aligned image stack built below.
tp_ids = tracks_df['t'].apply(lambda t: f'{t:05d}')
abs_first_z_um = tp_ids.map(absolute_first_z).astype(float) * VX_Z
tracks_df['z_um_abs'] = tracks_df[z_col] + abs_first_z_um

# Only nuclei actually inside the window's z-band for these timepoints - a
# clean spatial-temporal ROI rather than keeping tracks that merely pass
# through it.
before = len(tracks_df)
tracks_df = tracks_df[(tracks_df['z_um_abs'] >= Z_LO_UM) & (tracks_df['z_um_abs'] <= Z_HI_UM)].copy()
print(f'{len(tracks_df)}/{before} rows fall inside the z-window; '
      f'{tracks_df["track_id"].nunique()} distinct track(s).')

if tracks_df.empty:
    raise ValueError(
        f'No nuclei found in window {WINDOW_NAME} (t{T_START}-{T_END}, z{Z_LO_UM}-{Z_HI_UM}um absolute). '
        f'Check WINDOWS bounds against the absolute z printed below once frames are loaded.'
    )

Loaded 33 tracks, 273 rows in t70-80 from tracks_linked_c3_winB.csv (source=btrack)
273/273 rows fall inside the z-window; 33 distinct track(s).


In [75]:
# Load just this window's raw/label frames and align them to one shared
# absolute z-origin (the window's minimum absolute_first_z) - front-pad
# each frame by its own extra offset above that minimum, so local z-index k
# means the same real depth in every frame of the window. This is the fix
# for the drift seen when reusing one frame's local z-reading across a
# multi-timepoint window: each frame's *local* crop origin differs by a few
# to a dozen+ slices even after the debris correction, since that only
# removed debris - it never aligned frames to each other.
t_range = range(T_START, T_END + 1)
tp_ids_window = [f'{t:05d}' for t in t_range]

# Glob the full directories once, sorted - filenames sort in timepoint order
# (zero-padded), so index t directly picks out timepoint t. Avoids
# reconstructing per-timepoint filenames from a glob pattern, which breaks
# for label_glob (its wildcard covers more than just the digits - see
# raw_glob's "Dataset003_*_0000.tif" vs label_glob's
# "*_instances_reclassified.tif").
all_raw_files   = sorted(glob.glob(str(RAW_DIR   / RAW_GLOB)))
all_label_files = sorted(glob.glob(str(LABEL_DIR / LABEL_GLOB)))
assert len(all_raw_files) > T_END,   f'Only {len(all_raw_files)} raw files found, need index up to {T_END}'
assert len(all_label_files) > T_END, f'Only {len(all_label_files)} label files found, need index up to {T_END}'

raw_files   = [all_raw_files[t]   for t in t_range]
label_files = [all_label_files[t] for t in t_range]
for tp_id, rf, lf in zip(tp_ids_window, raw_files, label_files):
    assert tp_id in Path(rf).name, f'raw file {rf} does not match expected t={tp_id}'
    assert tp_id in Path(lf).name, f'label file {lf} does not match expected t={tp_id}'

print(f'Loading {len(raw_files)} raw/label frame(s) for window {WINDOW_NAME}...', flush=True)
raw_frames   = [imread(f) for f in raw_files]
label_frames = [imread(f) for f in label_files]

# The saved files on disk still start at each frame's OLD (pre-debris-fix)
# first_z - only crop_info.json/features.csv were patched, not the images
# themselves (see scripts/fix_debris_crop_offsets.py). Trim the debris-shift
# off the front here, identically to raw and labels, before computing the
# shared-origin alignment below - otherwise the padding math silently
# assumes a trim that was never actually applied to these arrays.
n_trimmed = 0
for i, tp_id in enumerate(tp_ids_window):
    shift = crop_info.get(tp_id, {}).get('debris_shift_slices')
    if shift:
        raw_frames[i]   = raw_frames[i][shift:]
        label_frames[i] = label_frames[i][shift:]
        n_trimmed += 1
if n_trimmed:
    print(f'Trimmed debris-inflated front padding off {n_trimmed} timepoint(s) in this window.')

abs_first_z_slices = {tp_id: absolute_first_z(tp_id) for tp_id in tp_ids_window}
shared_origin = min(abs_first_z_slices.values())
print('Absolute first_z per timepoint (slices):', abs_first_z_slices)
print(f'Shared window origin (absolute slice {shared_origin} = {shared_origin * VX_Z:.1f} um)')
print(f'To convert a z value read off the napari slider in THIS window to the '
      f'absolute WINDOWS[...]["z_lo_um"/"z_hi_um"] value: '
      f'absolute_um = napari_z_um + {shared_origin * VX_Z:.1f}')

def front_pad(frame, n):
    if n == 0:
        return frame
    pad_width = [(n, 0)] + [(0, 0)] * (frame.ndim - 1)
    return np.pad(frame, pad_width, mode='constant')

aligned_raw, aligned_label = [], []
for tp_id, rf, lf in zip(tp_ids_window, raw_frames, label_frames):
    extra = abs_first_z_slices[tp_id] - shared_origin
    aligned_raw.append(front_pad(rf, extra))
    aligned_label.append(front_pad(lf, extra))

max_z = max(f.shape[0] for f in aligned_raw)
def end_pad(frame, target_z):
    if frame.shape[0] == target_z:
        return frame
    pad_width = [(0, target_z - frame.shape[0])] + [(0, 0)] * (frame.ndim - 1)
    return np.pad(frame, pad_width, mode='constant')

raw_stack   = np.stack([end_pad(f, max_z) for f in aligned_raw])
label_stack = np.stack([end_pad(f, max_z) for f in aligned_label])

print(f'Raw stack:   {raw_stack.shape} ({raw_stack.nbytes / 1e9:.2f} GB)')
print(f'Label stack: {label_stack.shape} ({label_stack.nbytes / 1e9:.2f} GB)')
print(f'Window z-band in this aligned local frame: '
      f'[{(Z_LO_UM / VX_Z) - shared_origin:.0f}, {(Z_HI_UM / VX_Z) - shared_origin:.0f}] slices')

# tracks_df's z_um is still local-to-each-timepoint's *original* crop - shift
# to this aligned stack's local frame (relative to shared_origin) so points
# and images land on the same nuclei.
tracks_df[z_col + '_aligned'] = tracks_df['z_um_abs'] - shared_origin * VX_Z
z_col_aligned = z_col + '_aligned'

Loading 11 raw/label frame(s) for window B...
Trimmed debris-inflated front padding off 5 timepoint(s) in this window.
Absolute first_z per timepoint (slices): {'00070': 55, '00071': 54, '00072': 52, '00073': 49, '00074': 50, '00075': 52, '00076': 55, '00077': 56, '00078': 58, '00079': 58, '00080': 58}
Shared window origin (absolute slice 49 = 98.0 um)
To convert a z value read off the napari slider in THIS window to the absolute WINDOWS[...]["z_lo_um"/"z_hi_um"] value: absolute_um = napari_z_um + 98.0
Raw stack:   (11, 42, 1400, 1400) (1.81 GB)
Label stack: (11, 42, 1400, 1400) (1.81 GB)
Window z-band in this aligned local frame: [32, 41] slices


In [76]:
from src.tracking import build_track_label_stack

# build_track_label_stack indexes label_stack[t] as a LOCAL position (0..5
# for this window), but tracks_df['t'] holds real t (e.g. 45-50) everywhere
# else in this notebook (corrections/saves reference real t). Pass a
# local-t copy just for this call.
print('Building track-coloured label stack...', flush=True)
track_label_stack = build_track_label_stack(label_stack, tracks_df.assign(t=tracks_df['t'] - T_START))
print('Done.')

Building track-coloured label stack...
Done.


In [77]:
# Open napari — physical scale (T=1, Z, Y, X in µm). t here is offset to
# start at 0 (napari has no notion of the window's real T_START); the
# 'centroids'/'tracks' layers use t - T_START to match the loaded stack.
SCALE = (1, VX_Z, VX_Y, VX_X)

viewer = napari.Viewer(title=f'Window {WINDOW_NAME}: t{T_START}-{T_END}, z{Z_LO_UM}-{Z_HI_UM}um')

viewer.add_image(raw_stack, name='raw', scale=SCALE, colormap='gray', opacity=0.8)
viewer.add_labels(track_label_stack, name='track_labels', scale=SCALE, opacity=0.4)
viewer.add_labels(label_stack, name='instance_labels', scale=SCALE, opacity=0.25, blending='translucent')

t_local = tracks_df['t'] - T_START
centroid_coords = np.column_stack([t_local.values, tracks_df[z_col_aligned].values, tracks_df[y_col].values, tracks_df[x_col].values])
viewer.add_points(
    centroid_coords,
    name='centroids',
    features={'track_id': tracks_df['track_id'].values + 1},
    text={'string': '{track_id}', 'size': 12, 'color': 'black', 'anchor': 'center'},
    face_color='track_id',
    face_colormap='husl',
    border_color='black',
    border_width=0.05,
    size=4,
    opacity=0.9,
)

track_data = (
    tracks_df.assign(t_local=t_local)
    .sort_values(['track_id', 't_local'])
    [['track_id', 't_local', z_col_aligned, y_col, x_col]]
    .values
)
viewer.add_tracks(track_data, name='tracks', tail_length=10, head_length=0, tail_width=3)

print('napari layers: raw | track_labels | instance_labels | centroids | tracks')
print(f'Napari t=0 corresponds to real t={T_START}; t={T_END - T_START} corresponds to real t={T_END}.')

napari layers: raw | track_labels | instance_labels | centroids | tracks
Napari t=0 corresponds to real t=70; t=10 corresponds to real t=80.


In [72]:
# ─────────────────────────────────────────────────────────────
# Curation GUI — dock widget (identical to 04_curate_tracks.ipynb)
# ─────────────────────────────────────────────────────────────
from qtpy.QtWidgets import (
    QWidget, QVBoxLayout, QHBoxLayout, QGroupBox,
    QLabel, QLineEdit, QPushButton, QComboBox,
    QListWidget, QFileDialog, QSizePolicy
)
from qtpy.QtCore import Qt
from src.tracking import apply_corrections, check_duplicates


class CurationWidget(QWidget):
    def __init__(self, viewer, tracks_df, label_stack, build_fn,
                 z_col, y_col, x_col, tracks_dir, out_version, t_offset=0):
        super().__init__()
        self.viewer      = viewer
        self.tracks_df   = tracks_df
        self.label_stack = label_stack
        self.build_fn    = build_fn
        self.z_col, self.y_col, self.x_col = z_col, y_col, x_col
        self.tracks_dir  = tracks_dir
        self.out_version = out_version
        self.t_offset    = t_offset  # corrections below are entered in real t; internal df uses real t too
        self.corrections = []
        self._fields     = {}
        self._build_ui()

    def _build_ui(self):
        root = QVBoxLayout(self)

        type_box = QGroupBox('Add correction (timepoints are real t, not window-local)')
        type_layout = QVBoxLayout(type_box)
        self.type_combo = QComboBox()
        self.type_combo.addItems(['swap', 'reassign', 'break'])
        self.type_combo.currentTextChanged.connect(self._refresh_fields)
        type_layout.addWidget(self.type_combo)

        self.fields_layout = QVBoxLayout()
        type_layout.addLayout(self.fields_layout)
        self._refresh_fields('swap')

        add_btn = QPushButton('Add correction')
        add_btn.clicked.connect(self._add_correction)
        type_layout.addWidget(add_btn)
        root.addWidget(type_box)

        list_box = QGroupBox('Pending corrections')
        lbl = QVBoxLayout(list_box)
        self.corr_list = QListWidget()
        self.corr_list.setMaximumHeight(160)
        lbl.addWidget(self.corr_list)
        rm_btn = QPushButton('Remove selected')
        rm_btn.clicked.connect(self._remove_correction)
        lbl.addWidget(rm_btn)
        root.addWidget(list_box)

        self.preview_btn = QPushButton('Preview in napari')
        self.preview_btn.clicked.connect(self._preview)
        root.addWidget(self.preview_btn)

        self.save_csv_btn = QPushButton('Save corrected CSV')
        self.save_csv_btn.clicked.connect(self._save_csv)
        root.addWidget(self.save_csv_btn)

        self.save_tif_btn = QPushButton('Save corrected label TIFF')
        self.save_tif_btn.clicked.connect(self._save_tiff)
        root.addWidget(self.save_tif_btn)

        self.status = QLabel('')
        self.status.setWordWrap(True)
        root.addWidget(self.status)
        root.addStretch()

    FIELD_SPECS = {
        'swap':     [('t_from', 'Start timepoint (real t)'), ('id_a', 'Track ID A (as shown)'), ('id_b', 'Track ID B (as shown)')],
        'reassign': [('t',      'Start timepoint (real t)'), ('from_id', 'From ID (as shown)'), ('to_id', 'To ID (as shown)')],
        'break':    [('track_id', 'Track ID (as shown)'), ('t_break', 'Break at timepoint (real t)')],
    }
    _TRACK_ID_KEYS = {'id_a', 'id_b', 'from_id', 'to_id', 'track_id'}

    def _refresh_fields(self, ctype):
        while self.fields_layout.count():
            item = self.fields_layout.takeAt(0)
            if item.widget():
                item.widget().deleteLater()
        self._fields = {}
        for key, label in self.FIELD_SPECS.get(ctype, []):
            row = QHBoxLayout()
            row.addWidget(QLabel(f'{label}:'))
            edit = QLineEdit()
            edit.setPlaceholderText('int')
            row.addWidget(edit)
            self._fields[key] = edit
            container = QWidget()
            container.setLayout(row)
            self.fields_layout.addWidget(container)

    def _add_correction(self):
        ctype = self.type_combo.currentText()
        try:
            corr = {'type': ctype}
            for key, edit in self._fields.items():
                val = int(edit.text())
                if key in self._TRACK_ID_KEYS:
                    val -= 1  # displayed IDs are track_id+1; CSV stores raw track_id
                corr[key] = val
        except ValueError:
            self.status.setText('All fields must be integers.')
            return
        self.corrections.append(corr)
        self.corr_list.addItem(str(corr))
        for e in self._fields.values():
            e.clear()
        self.status.setText(f'{len(self.corrections)} correction(s) pending.')

    def _remove_correction(self):
        row = self.corr_list.currentRow()
        if row >= 0:
            self.corr_list.takeItem(row)
            self.corrections.pop(row)
        self.status.setText(f'{len(self.corrections)} correction(s) pending.')

    def _get_corrected_df(self):
        return apply_corrections(self.tracks_df, self.corrections)

    def _preview(self):
        self.status.setText('Building preview...')
        try:
            cdf = self._get_corrected_df()
            dups = check_duplicates(cdf)
            if not dups.empty:
                disp = dups.copy()
                disp['track_id'] = disp['track_id'] + 1
                disp = disp.rename(columns={'track_id': 'track_id (displayed)'})
                msg = f'{len(dups)} duplicate (track_id, t) pair(s) found — fix before saving:\n'
                msg += disp.head(10).to_string(index=False)
                self.status.setText(msg)
                return

            cdf_local = cdf.assign(t=cdf['t'] - self.t_offset)
            self.viewer.layers['track_labels'].data = self.build_fn(self.label_stack, cdf_local)
            self.viewer.layers['track_labels'].refresh()

            pts = self.viewer.layers['centroids']
            t_local = cdf['t'] - self.t_offset
            pts.data = np.column_stack([t_local.values, cdf[self.z_col].values, cdf[self.y_col].values, cdf[self.x_col].values])
            pts.features = {'track_id': cdf['track_id'].values + 1}
            pts.face_color = 'track_id'

            n_before = self.tracks_df['track_id'].nunique()
            n_after  = cdf['track_id'].nunique()
            self.status.setText(f'Preview updated. Tracks: {n_before} -> {n_after}')
        except Exception as e:
            self.status.setText(f'Error: {e}')

    def _save_csv(self):
        default = str(self.tracks_dir / f'tracks_{self.out_version}.csv')
        path, _ = QFileDialog.getSaveFileName(self, 'Save corrected CSV', default, 'CSV files (*.csv)')
        if not path:
            return
        try:
            cdf = self._get_corrected_df()
            dups = check_duplicates(cdf)
            if not dups.empty:
                self.status.setText(f'Save blocked: {len(dups)} duplicate (track_id, t) pair(s). Run preview to see details.')
                return
            cdf.to_csv(path, index=False)
            self.status.setText(f'CSV saved: {path}')
        except Exception as e:
            self.status.setText(f'Error: {e}')

    def _save_tiff(self):
        default = str(self.tracks_dir / f'track_labels_{self.out_version}.tif')
        path, _ = QFileDialog.getSaveFileName(self, 'Save corrected label TIFF', default, 'TIFF files (*.tif *.tiff)')
        if not path:
            return
        self.status.setText('Building corrected label stack...')
        try:
            cdf = self._get_corrected_df()
            dups = check_duplicates(cdf)
            if not dups.empty:
                self.status.setText(f'Save blocked: {len(dups)} duplicate (track_id, t) pair(s). Run preview to see details.')
                return
            cdf_local = cdf.assign(t=cdf['t'] - self.t_offset)
            imsave(path, self.build_fn(self.label_stack, cdf_local))
            self.status.setText(f'TIFF saved: {path}')
        except Exception as e:
            self.status.setText(f'Error: {e}')


curation_widget = CurationWidget(
    viewer      = viewer,
    tracks_df   = tracks_df,
    label_stack = label_stack,
    build_fn    = build_track_label_stack,
    z_col=z_col_aligned, y_col=y_col, x_col=x_col,
    tracks_dir  = TRACKS_DIR,
    out_version = OUT_VERSION,
    t_offset    = T_START,
)
viewer.window.add_dock_widget(curation_widget, name=f'Track Curation — Window {WINDOW_NAME}', area='right')
print('Curation widget docked.')

Curation widget docked.


/home/elysse/miniforge3/envs/napari_env/lib/python3.13/site-packages/skimage/_shared/utils.py:386: UserWarning: /mnt/md0/elysse/nnUNet/inference/Dataset003_icm_te/250914_stack5/results/tracks/btrack/track_labels_linked_c3_winB.tif is a low contrast image
  return func(*args, **kwargs)


## Editing label shapes (not just track IDs)

The curation widget above only rewrites *track_id* assignments — it never
touches the underlying segmentation. To fix an actual label's shape (merge,
split, erase, redraw a boundary), edit the `instance_labels` layer directly
in napari: select it in the layers list, then use the paintbrush (`2`),
eraser, fill bucket (`3`), and label picker (`l`) like any napari Labels
layer.

Run the cell below whenever you want to save what you've painted. It writes
into a **sibling directory**, not over the originals in `LABEL_DIR` — review
the output there and copy filenames over the originals yourself once you're
happy with them (or repoint `LABEL_DIR` at it), since other pipeline steps
(feature extraction, tracking) read `LABEL_DIR` as-is.

In [ ]:
# Save hand-edited instance-label SHAPES back out as per-timepoint TIFFs.
#
# Reverses the two alignment steps applied when label_stack was built above:
# the per-frame front-pad used to share one z-origin across the window, and
# the end-pad used to give every frame the same z-depth. Also re-adds any
# debris-trim slices removed from the front there, since crop_info.json was
# patched for the debris fix but the on-disk label files were not (see
# scripts/fix_debris_crop_offsets.py) - skipping this would shift the saved
# file's z-origin away from what every other pipeline step expects.
EDITED_LABEL_DIR = LABEL_DIR.parent / f'{LABEL_DIR.name}_edited_win{WINDOW_NAME}'
EDITED_LABEL_DIR.mkdir(parents=True, exist_ok=True)

edited_stack = viewer.layers['instance_labels'].data

saved, changed = [], []
for i, tp_id in enumerate(tp_ids_window):
    extra     = abs_first_z_slices[tp_id] - shared_origin
    trimmed_z = label_frames[i].shape[0]  # depth after debris-trim, before front/end-pad
    frame     = edited_stack[i, extra:extra + trimmed_z]

    shift = crop_info.get(tp_id, {}).get('debris_shift_slices', 0)
    if shift:
        frame = np.pad(frame, [(shift, 0), (0, 0), (0, 0)], mode='constant')

    frame = frame.astype(label_frames[i].dtype, copy=False)
    out_path = EDITED_LABEL_DIR / Path(label_files[i]).name
    imsave(out_path, frame, check_contrast=False)
    saved.append(out_path)

    if not np.array_equal(frame, imread(label_files[i])):
        changed.append(tp_id)

print(f'Saved {len(saved)} frame(s) to {EDITED_LABEL_DIR}')
print(f'Differ from the original on-disk labels: {changed or "none"}')
print(f'\nReview against {LABEL_DIR} - once approved, copy the changed filename(s) '
      f'over the originals there to promote them into the pipeline.')